In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv('Swan Consulting 1 - Project Data.csv')

In [3]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Count', 'State', 'Country', 'CustomerID'])
df_clean = df_clean.drop(columns= ['Lat Long', 'City','Zip Code'])
df_clean = df_clean.drop(columns= ['Churn Label'])
df_clean = df_clean.drop(columns= ['Churn Reason'])
df_clean = df_clean.drop(columns= ['Latitude', 'Longitude'])

In [4]:
df_clean['Gender'] = df_clean['Gender'].map({'Male':0, 'Female':1})
df_clean['Senior Citizen'] = df_clean['Senior Citizen'].map({'No':0, 'Yes':1})
df_clean['Partner'] = df_clean['Partner'].map({'No':0, 'Yes':1})
df_clean['Dependents'] = df_clean['Dependents'].map({'No':0, 'Yes':1})
df_clean['Phone Service'] = df_clean['Phone Service'].map({'No':0, 'Yes':1})
df_clean['Paperless Billing'] = df_clean['Paperless Billing'].map({'No':0, 'Yes':1})

In [5]:
df_clean = pd.get_dummies(df_clean, columns=['Multiple Lines'], drop_first=True)
df_clean = pd.get_dummies(df_clean, columns=[
    'Internet Service',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies',
    'Contract',
    'Payment Method'
], drop_first=True)

In [6]:
df_clean = df_clean.astype({col: int for col in df_clean.select_dtypes(include='bool').columns})
df_clean[pd.to_numeric(df_clean['Total Charges'], errors='coerce').isna()][['Total Charges', 'Tenure Months']]
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce').fillna(0)

In [7]:
X = df_clean.drop(columns=['Churn Value'])
y = df_clean['Churn Value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid = {
    'scaler': [StandardScaler(), RobustScaler()],
    'model__C': [0.01, 0.1, 1, 10]
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)

{'model__C': 0.1, 'scaler': RobustScaler()}


In [9]:
scoring = ['accuracy', 'precision', 'recall', 'f1']
cv_results = cross_validate(grid_search.best_estimator_, X_train, y_train, cv=kfold, scoring=scoring)

for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.3f} +/- {scores.std():.3f}  {scores.round(3)}")

accuracy: 0.760 +/- 0.017  [0.76  0.744 0.749 0.752 0.792]
precision: 0.526 +/- 0.022  [0.526 0.507 0.512 0.517 0.568]
recall: 0.804 +/- 0.027  [0.827 0.762 0.803 0.789 0.84 ]
f1: 0.636 +/- 0.024  [0.643 0.609 0.625 0.624 0.678]


In [10]:
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1009
           1       0.54      0.80      0.65       400

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.80      0.75      0.76      1409



In [11]:
model = grid_search.best_estimator_.named_steps['model']
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coefficients)

                                   Feature  Coefficient
11            Internet Service_Fiber optic     0.562745
6                        Paperless Billing     0.346725
28         Payment Method_Electronic check     0.333252
7                          Monthly Charges     0.260198
10                      Multiple Lines_Yes     0.228523
8                            Total Charges     0.206724
9          Multiple Lines_No phone service     0.184875
22                        Streaming TV_Yes     0.172663
2                                  Partner     0.168501
24                    Streaming Movies_Yes     0.157910
1                           Senior Citizen     0.126229
0                                   Gender     0.025403
29             Payment Method_Mailed check    -0.007723
18                   Device Protection_Yes    -0.028375
16                       Online Backup_Yes    -0.103519
13     Online Security_No internet service    -0.111076
23    Streaming Movies_No internet service    -0